# Fine-tune the folder assistant (Qwen3.5-4B) on Kaggle

The same as `finetune_colab.ipynb`, for when Colab's free GPU limit is used up. Kaggle gives about 30 free
GPU hours a week. The project code is already inside this notebook. You only add a file when a run stopped
halfway: the newest `local-assistant-progress.zip` it gave you.

**Once:** verify your phone number at kaggle.com > Settings. Kaggle needs it before it gives you a GPU and internet.

**Setup:**
1. kaggle.com > **Create > New Notebook**. In the notebook: **File > Import Notebook** and choose this file.
2. Right panel, **Session options**: Accelerator **GPU T4 x2** (not P100: Unsloth needs a newer GPU), Internet **On**.
3. Only when continuing a run that stopped: right panel, **Input > Add Input > Upload**, drop the newest
   `local-assistant-progress.zip` from that run, give it any name and click Create. Progress zips from rounds
   before r9 are ignored, because their requests are different.
4. Click **Run All** at the top, or run the cells from top to bottom. Keep the tab open.

What happens: Part A makes 1,200 requests about 300 fake folders, and the teacher (Qwen3.5-9B) answers them on both
GPUs at once (about 4 to 5 hours, with a progress zip after every 300). A4 checks the answers, splits them and
counts them, and gives you `training-data.zip`: keep it, the report needs it. Part B trains (about 2 hours) and
makes the `.gguf` file. B7 then shows download links for the model in 300 MB parts. Kaggle deletes the files
when the session ends, so download them before you close the tab.

In [ ]:
# Step 0: the project code (101 KB zip), put here by colab/embed_code.py. Just run it.
CODE_ZIP = """
...
"""


In [ ]:
# Step 1: unpack the project code and take the data from your old progress zip (in the notebook's Input)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # "T4 x2" has two GPUs; one is enough and Unsloth expects one
import sys, io, glob, json, base64, socket, zipfile, shutil, subprocess
from IPython.display import FileLink, display
WORK = "/kaggle/working"   # small files to download (Kaggle keeps at most 20 GB here)
TMP = "/kaggle/temp"       # big temporary files: the merged model and the unquantized .gguf
PROJECT = f"{WORK}/local-assistant"
DATA = ("tasks.jsonl", "drafts.jsonl", "rejected.jsonl")
SCENARIOS = 300            # fake folders, 4 requests each. A4 says when to raise it to get more training data.

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().replace("\n", ", ")
except FileNotFoundError:
    gpu = "no GPU"
if "T4" not in gpu:
    raise RuntimeError(f"This session has {gpu}. Choose GPU T4 x2 in Session options (right panel), then run again.")
try:
    socket.create_connection(("pypi.org", 443), timeout=10).close()
except OSError:
    raise RuntimeError("This session has no internet. Right panel > Session options > Internet: On. If Kaggle won't let "
                       "you, verify your phone number first (profile picture > Settings > Phone verification). "
                       "Then run again from the top.") from None
os.makedirs(TMP, exist_ok=True)
if not os.path.exists(f"{PROJECT}/assistant"):
    zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP))).extractall(WORK)
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)


def link(path):
    """A download link for a file in /kaggle/working (also listed under Output in the right panel)."""
    here = os.getcwd()
    os.chdir(WORK)
    display(FileLink(os.path.relpath(path, WORK)))
    os.chdir(here)


def save_progress():
    """Zips the project with all data made so far and shows a download link. If the session stops, add the
    newest one as Input instead of the old one and run again: it continues where it stopped."""
    shutil.make_archive(f"{WORK}/local-assistant-progress", "zip", WORK, "local-assistant")
    link(f"{WORK}/local-assistant-progress.zip")


def lines(name):
    path = f"dataset/{name}"
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()] if os.path.exists(path) else []


def tasks_left():
    done = {r["meta"]["task_id"] for r in lines("drafts.jsonl") + lines("rejected.jsonl")}
    return sum(1 for t in lines("tasks.jsonl") if t["task_id"] not in done)


def scenarios():
    return len({t["scenario_id"] for t in lines("tasks.jsonl")})


def count_lines(name):
    """Lines in a data file without reading them as JSON: safe while the teacher is still writing to it."""
    path = f"dataset/{name}"
    return open(path, encoding="utf-8").read().count("\n") if os.path.exists(path) else 0


# Kaggle unpacks some uploaded zips and keeps others as they are, so look for both
found = {}
for folder in glob.glob("/kaggle/input/**/dataset/", recursive=True):
    found[folder] = {f: open(folder + f, "rb").read() for f in DATA if os.path.exists(folder + f)}
for path in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    with zipfile.ZipFile(path) as z:
        found[path] = {f: z.read(n) for n in z.namelist() for f in DATA if n.endswith("dataset/" + f)}
found = {k: v for k, v in found.items() if v}
for k in [k for k, v in found.items() if b'"one_file"' not in v.get("tasks.jsonl", b"")]:
    print("Ignored", k, "- it is from a round before r9, whose requests were different")
    del found[k]
if found and not any(os.path.exists(f"dataset/{f}") for f in DATA):
    source, data = max(found.items(), key=lambda kv: kv[1].get("drafts.jsonl", b"").count(b"\n"))
    for f, content in data.items():
        with open(f"dataset/{f}", "wb") as out:
            out.write(content)
    print("Took", ", ".join(data), "from", source)

PART_A = not lines("tasks.jsonl") or tasks_left() > 0 or scenarios() < SCENARIOS
if not lines("drafts.jsonl"):
    print("No progress zip from this round in Input, so Part A makes all answers first (about 4 to 5 hours).")
else:
    print(f"From your progress zip: {len(lines('tasks.jsonl'))} tasks, {len(lines('drafts.jsonl'))} drafts, "
          f"{len(lines('rejected.jsonl'))} rejected.",
          f"Part A drafts the missing answers first (about 1 hour per 300)." if PART_A else "Part A is skipped.")
print("GPU:", gpu, "| Internet: on | Ready:", os.getcwd())

## Part A: make the training data (skipped when your progress zip already has every answer)

In [ ]:
# A1: install Ollama and the teacher model (about 5 minutes). With T4 x2, one Ollama server per GPU drafts
# twice as fast.
TEACHER = "qwen3.5:9b"          # fits the T4
HOSTS = [f"127.0.0.1:{11434 + i}" for i in range(max(1, gpu.count("T4")))]
if PART_A:
    !apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    if not shutil.which("ollama"):
        raise RuntimeError("Ollama did not install. See the error above.")
    import time
    for i, host in enumerate(HOSTS):
        subprocess.Popen(["ollama", "serve"], env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i), "OLLAMA_HOST": host},
                         stdout=open(f"{TMP}/ollama{i}.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
    !ollama pull {TEACHER}
    !pip install -q ollama pypdf python-docx pillow send2trash
    print(len(HOSTS), "Ollama server(s), one per GPU")
else:
    print("Skipped")


In [ ]:
# A2: make the fake folders and requests: SCENARIOS folders x 4 requests (300 gives 1,200 tasks). Raising
# SCENARIOS later keeps the first folders and requests the same, so only the new ones get drafted.
if PART_A and scenarios() < SCENARIOS:
    !python dataset/make_scenarios.py --scenarios {SCENARIOS} --requests 4
print(len(lines("tasks.jsonl")), "tasks from", scenarios(), "folders")


In [ ]:
# A3: the teacher answers every task that has no answer yet, 150 per GPU at a time (about 1 hour), with a new
# progress zip after each round. A status line every minute.
while PART_A and tasks_left():
    before = tasks_left()
    procs = [subprocess.Popen([sys.executable, "dataset/draft_answers.py", "--teacher", TEACHER, "--limit", "150",
                               "--shard", f"{i}/{len(HOSTS)}"], env={**os.environ, "OLLAMA_HOST": host},
                              stdout=open(f"{TMP}/draft{i}.log", "a"), stderr=subprocess.STDOUT)
             for i, host in enumerate(HOSTS)]
    while any(p.poll() is None for p in procs):
        time.sleep(60)
        print(f"{count_lines('drafts.jsonl')} answers kept, {count_lines('rejected.jsonl')} rejected", flush=True)
    save_progress()
    if tasks_left() == before:
        !tail -n 20 {TMP}/draft0.log
        raise RuntimeError("No new answers were drafted. See the error above.")
    print(tasks_left(), "tasks left")
print(tasks_left(), "tasks left")


In [ ]:
# A4: check the answers again with today's rules (and keep what can be kept from rejected ones), split them
# into train / validation / test, count them, and give you training-data.zip
sys.path.insert(0, "dataset")
from pathlib import Path
from validate import check, drop_bad_changes
from stats import count
drafts, rejected = lines("drafts.jsonl"), [r for r in lines("rejected.jsonl") if "messages" in r]
kept = []
for d in drafts + rejected:
    d = {k: v for k, v in d.items() if k != "errors"}
    if not check(d):
        kept.append(d)
    elif (fixed := drop_bad_changes(d)[0]):  # drop only the changes that break a rule
        kept.append(fixed)
with open("dataset/reviewed.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(d, ensure_ascii=False) + "\n" for d in kept)
print(f"{len(kept)} of {len(drafts) + len(rejected)} answers are used "
      f"({sum('dropped' in d.get('meta', {}) for d in kept)} without a few changes that broke a rule)")
if len(kept) < 60:
    raise RuntimeError(f"Only {len(kept)} drafts pass: too few to train a better model. Stop here and send this number.")
!python dataset/make_questions.py
!python dataset/split.py
!python dataset/stats.py --md dataset/data_stats.md
decisions = sum(count(Path(f"dataset/{s}.jsonl"))["file decisions"] for s in ("train", "val", "test"))
shutil.make_archive(f"{WORK}/training-data", "zip", PROJECT, "dataset")
link(f"{WORK}/training-data.zip")
save_progress()
if decisions < 5001:
    print(f"Only {decisions:,} file decisions: under the 5,001 the pre-assessment form needs. Set SCENARIOS = "
          f"{SCENARIOS + 100} in Step 1 and run Step 1 and Part A again; only the new requests are drafted.")


In [ ]:
# A5: stop Ollama so the GPU is free for training
!pkill -f "ollama serve"; sleep 3; nvidia-smi --query-gpu=memory.used --format=csv

## Part B: fine-tune the 4B

In [ ]:
# B1: install Unsloth (copied from Unsloth's official Qwen3.5 notebook, about 5 minutes)
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# B2: load Qwen3.5-4B and add a LoRA adapter (only about 1% of the weights are trained)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less wasted GPU memory
from unsloth import FastVisionModel   # Qwen3.5 is a vision-language model; we train only its language part
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    # Unsloth advises 16-bit LoRA for Qwen3.5, but that needs more memory than the free T4's 15 GB, so the T4 uses 4-bit
    load_in_4bit = torch.cuda.get_device_properties(0).total_memory < 20 * 2**30,
    use_gradient_checkpointing = "unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 3407,
)
tok = getattr(processor, "tokenizer", processor)

In [ ]:
# B3: turn the data into training examples, with the model's own chat template ("no thinking", like the app)
from assistant.schema import TOOLS
from train import sft_data

TRAIN, VAL = "dataset/train.jsonl", "dataset/val.jsonl"
SMOKE_TEST = not os.path.exists(TRAIN)
if SMOKE_TEST:
    TRAIN = VAL = "dataset/seed_examples.jsonl"
    print("No training data yet: 5-step test run on the 7 hand-written examples.")

template = tok if getattr(tok, "chat_template", None) else processor
def apply_template(messages, add_generation_prompt, tools=TOOLS):  # questions have their own tools
    return template.apply_chat_template(messages, tools=tools, tokenize=False,
                                        add_generation_prompt=add_generation_prompt, enable_thinking=False)

MAX_LEN = 6144
train, skipped = sft_data.build(TRAIN, tok, apply_template, MAX_LEN)
val, _ = sft_data.build(VAL, tok, apply_template, MAX_LEN)
lengths = sorted(len(e["input_ids"]) for e in train)
print(f"train {len(train)} examples, val {len(val)}, skipped {skipped}")
print(f"tokens per example: median {lengths[len(lengths)//2]}, max {lengths[-1]}")
ex = train[1]
cut = next(i for i, l in enumerate(ex["labels"]) if l != -100)
print("...", tok.decode(ex["input_ids"][cut-60:cut]), "\n==== THE MODEL LEARNS THIS PART ====\n", tok.decode(ex["input_ids"][cut:]))

In [ ]:
# B4: train (about 2 hours with 1,200 tasks; 5 minutes for the test run)
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# On GPUs without bfloat16 (the T4) Unsloth trains Qwen3.5 in float32, because float16 gives NaN errors in its layers
forced32 = os.environ.get("UNSLOTH_FORCE_FLOAT32") == "1"
bf16 = torch.cuda.is_bf16_supported() and not forced32
FastVisionModel.for_training(model)
trainer = Trainer(
    model = model,
    args = TrainingArguments(
        output_dir = f"{TMP}/outputs",
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,      # long examples: 8 at once would run out of GPU memory
        gradient_accumulation_steps = 8,
        num_train_epochs = 1 if len(train) > 1000 else 2,   # with lots of data, once through is enough
        max_steps = 5 if SMOKE_TEST else -1,
        learning_rate = 2e-4,
        warmup_steps = 0.05,                 # under 1 means a fraction: warm up for 5% of the steps
        lr_scheduler_type = "linear",
        weight_decay = 0.001,
        optim = "adamw_8bit",
        logging_steps = 1 if SMOKE_TEST else 10,
        eval_strategy = "no" if SMOKE_TEST else "epoch",
        prediction_loss_only = True,         # evaluation needs only the loss, not the logits for every token
        save_strategy = "no",
        bf16 = bf16, fp16 = not bf16 and not forced32,
        report_to = "none",
        seed = 3407,
        remove_unused_columns = False,
    ),
    train_dataset = Dataset.from_list(train),
    eval_dataset = Dataset.from_list(val),
    data_collator = DataCollatorForSeq2Seq(tok, padding = True, label_pad_token_id = -100),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} minutes, peak GPU memory {torch.cuda.max_memory_reserved() / 2**30:.1f} GB")

In [ ]:
# B5: quick look at what the trained model answers for 3 validation requests
import json
FastVisionModel.for_inference(model)
records = [json.loads(l) for l in open(VAL, encoding="utf-8") if l.strip()][:3]
for rec in records:
    prompt, target = sft_data.examples(rec)[-1]
    inputs = tok(apply_template(prompt, True, sft_data.tools_for(rec)), return_tensors="pt",
                 add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=800, do_sample=False)
    print("USER:", [m["content"] for m in prompt if m["role"] == "user"][-1])
    print("MODEL:", tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False), "\n" + "-" * 80)

In [ ]:
# B6: make the .gguf file for your PC (about 10 to 20 minutes)
NAME = "assistant-4b" + ("-test" if SMOKE_TEST else "")
model.save_pretrained(f"{TMP}/{NAME}-lora")              # the small adapter, kept as a backup
processor.save_pretrained(f"{TMP}/{NAME}-lora")
shutil.make_archive(f"{WORK}/{NAME}-lora", "zip", TMP, f"{NAME}-lora")
model.save_pretrained_gguf(f"{TMP}/gguf", processor, quantization_method = "q4_k_m")
!ls -lh {TMP}/gguf*/

In [ ]:
# B7: cut the .gguf (about 2.7 GB) into 300 MB parts and show download links. Click every link (or use Output
# in the right panel) and wait until all downloads finish before you close the tab.
import hashlib
src = next(p for p in glob.glob(f"{TMP}/gguf*/*.gguf") if "q4_k_m" in p.lower() and "mmproj" not in p)
for p in glob.glob(f"{TMP}/gguf*/*"):
    if p != src and os.path.isfile(p):
        os.remove(p)                                     # free disk space: only the q4_k_m file is needed
os.makedirs(f"{WORK}/parts", exist_ok=True)
sha = hashlib.sha256()
with open(src, "rb") as f:
    n = 0
    while chunk := f.read(300 * 2**20):
        sha.update(chunk)
        n += 1
        with open(f"{WORK}/parts/{os.path.basename(src)}.part{n:02d}", "wb") as out:
            out.write(chunk)
with open(f"{WORK}/parts/{os.path.basename(src)}.sha256.txt", "w") as out:
    out.write(f"{sha.hexdigest()}  {os.path.basename(src)}  {os.path.getsize(src)} bytes  {n} parts\n")
print(open(f"{WORK}/parts/{os.path.basename(src)}.sha256.txt").read())
shutil.copy("dataset/test.jsonl", f"{WORK}/test.jsonl")   # held-out test requests, for the comparison on your PC
for path in sorted(glob.glob(f"{WORK}/parts/*")) + [f"{WORK}/test.jsonl", f"{WORK}/{NAME}-lora.zip"]:
    link(path)
if os.path.exists(f"{WORK}/training-data.zip"):
    link(f"{WORK}/training-data.zip")                     # the data the model learned from: keep it for the report

## Run the fine-tuned model on your PC

1. Move the downloaded `.partNN` files and the `.sha256.txt` file into `C:\Users\kerim\Projects\local-assistant\models\`,
   `test.jsonl` into `dataset\`, and `training-data.zip` into `results\` (it holds the training data and its counts,
   `dataset/data_stats.md`, for the report). Then, in `local-assistant`, join the parts into one `.gguf` file:
   `python train/join_parts.py models`
2. Start it with llama.cpp (already on your PC), with <file> replaced by the file's name:
   ```
   C:\Users\kerim\Projects\tools\llama.cpp\bin\llama-server.exe -m C:\Users\kerim\Projects\local-assistant\models\<file>.gguf --jinja -ngl 99 -c 8192 --port 8080
   ```
3. In another terminal in `local-assistant`:
   ```
   .venv\Scripts\activate
   set LLM_BACKEND=llama-server
   python cli.py --base sandbox
   python bench/evaluate.py dataset/test.jsonl
   ```
   Run the same evaluation with `set LLM_BACKEND=ollama` to compare the original 4B with the fine-tuned one.